# Pertes d'un MOSFET seul

Calcul des pertes et de la température de jonction d'**un MOSFET unique**, à partir des
données constructeur du composant et de son driver.

**Mode d'emploi — 3 étapes :**

1. Exécuter la cellule de configuration (`Kernel → Restart & Run All` fait tout d'un coup).
2. Choisir le MOSFET, le driver et le point de fonctionnement dans le panneau.
3. Cliquer sur **Calculer**.

> Pour un bras de pont (recouvrement croisé entre high side et low side), voir le notebook
> demi-pont : ici la diode qui se recouvre est celle du MOSFET lui-même, ou une diode
> externe dont on saisit le $Q_{rr}$ à la main.

In [4]:
# --- Configuration : à exécuter en premier -----------------------------------
import sys
from pathlib import Path

# Remonte jusqu'à la racine du projet (le dossier qui contient DATABASE/)
ROOT = Path.cwd()
while not (ROOT / "DATABASE").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import base64
import ipywidgets as W
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output, display

from DATABASE.db_driver_mosfet import DRIVER_LIBRARY, load_driver
from DATABASE.db_mosfet import MOSFET_LIBRARY, load_mosfet
from SRC.MOSFET.mosfet_loss import (
    OPERATING_POINT,
    loss_single_mosfet_at_temp,
    loss_thermal_iteration,
)
from SRC.MOSFET.mosfet_plot import loss_table, plot_loss_breakdown, plot_thermal_iteration

#%matplotlib inline

print(f"Racine du projet : {ROOT}")
print(f"MOSFET disponibles : {', '.join(sorted(MOSFET_LIBRARY))}")
print(f"Drivers disponibles : {', '.join(sorted(DRIVER_LIBRARY))}")

Racine du projet : c:\Users\tomro\Documents\MyCalculator_V3
MOSFET disponibles : BSC016N06NS
Drivers disponibles : UCC27714


---
## 1. Figure de référence — la charge de grille

Tous les temps de commutation du modèle sortent de ce diagramme. Les charges
$Q_{g(th)}$, $Q_{gs}$, $Q_{gd}$ et le plateau Miller sont ce qui pilote la vitesse
de commutation, donc les pertes.

Choisis l'image à afficher dans la liste (tout ce qui est dans `DOCUMENTS/`).

In [5]:
# --- Image de référence ------------------------------------------------------
_MIME = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
         ".webp": "image/webp", ".gif": "image/gif", ".svg": "image/svg+xml"}

_images = sorted(
    (p for p in (ROOT / "DOCUMENTS").rglob("*") if p.suffix.lower() in _MIME),
    key=lambda p: p.name,
)


def show_image(path, max_width=720):
    """Affiche une image en l'embarquant en base64 (marche aussi pour le .webp)."""
    path = Path(path)
    if not path.exists():
        return HTML(f"<p style='color:#d03b3b'>Image introuvable : {path}</p>")
    mime = _MIME.get(path.suffix.lower(), "image/png")
    b64 = base64.b64encode(path.read_bytes()).decode()
    return HTML(
        f"<img src='data:{mime};base64,{b64}' "
        f"style='max-width:{max_width}px;width:100%;border-radius:6px'>"
    )


# Par défaut : le diagramme de charge de grille
_default = next((p for p in _images if "gate_charge" in p.name), _images[0] if _images else None)

image_dd = W.Dropdown(
    options=[(p.relative_to(ROOT).as_posix(), p) for p in _images],
    value=_default,
    description="Image :",
    style={"description_width": "80px"},
    layout=W.Layout(width="620px"),
)
image_out = W.Output()


def _refresh_image(_=None):
    with image_out:
        clear_output(wait=True)
        display(show_image(image_dd.value))


image_dd.observe(_refresh_image, names="value")
_refresh_image()
display(W.VBox([image_dd, image_out]))

---
## 2. Les formules

### 2.1 Boucle de grille — d'où viennent les temps

$$R_{G,tot} = R_{drv} + R_{ext} + R_{g,int}
\qquad
I_G = \frac{V_{drive} - V_{gate}}{R_{G,tot}}
\quad\text{(borné au courant crête du driver)}$$

L'amorçage se décompose en deux sous-intervalles, et le blocage en est le miroir :

| Sous-intervalle | Charge déplacée | Tension de grille | Ce qui bouge |
|---|---|---|---|
| $t_{ri}$ | $Q_{gs} - Q_{g(th)}$ | $\tfrac12(V_{th}+V_{pl})$ | le **courant** monte, $V_{ds}$ reste haute |
| $t_{fv}$ | $Q_{gd}$ | $V_{pl}$ (plateau) | la **tension** descend |
| $t_{rv}$ | $Q_{gd}$ | $V_{pl}$ | la tension remonte |
| $t_{fi}$ | $Q_{gs} - Q_{g(th)}$ | $\tfrac12(V_{th}+V_{pl})$ | le courant descend |

$$t = \frac{Q_{\text{région}}}{I_G}
\qquad
t_{on} = t_{ri} + t_{fv}
\qquad
t_{off} = t_{rv} + t_{fi}$$

### 2.2 Les six pertes

$$\boxed{P_{cond} = R_{DS(on)}(T_j)\; I_{rms}^2}
\qquad
R_{DS(on)}(T_j) = R_{25}\left[1 + \alpha_R (T_j - 25)\right]$$

$$\boxed{P_{sw} = \tfrac12\left(V_{on} I_{on} t_{on} + V_{off} I_{off} t_{off}\right) f_{sw}}$$

$$\boxed{P_{oss} = \tfrac12\, C_{oss,er}(V_{on})\, V_{on}^2\, f_{sw}}
\qquad
C_{oss,er}(V) = \frac{2}{V^2}\int_0^{V} C_{oss}(v)\, v\, dv$$

$$\boxed{P_{body} = \underbrace{V_F I_{body} D_{body}}_{\text{conduction temps mort}}
+ \underbrace{Q_{rr}\, V_{on}\, f_{sw}}_{\text{recouvrement}}}$$

$Q_{rr}$ n'est **pas** une propriété de la diode seule : c'est la charge que la
commutation arrive à extraire. Les trois conditions du point de test datasheet la
déplacent, d'où la loi à trois facteurs :

$$\boxed{Q_{rr} = Q_{rr,typ}
\left(\frac{di/dt}{(di/dt)_{ref}}\right)^{a}
\left(\frac{V_R}{V_{R,ref}}\right)^{b}
\left(\frac{I_F}{I_{F,ref}}\right)^{c}}
\qquad a = b = c = \tfrac12 \text{ par défaut}$$

| Facteur | Pourquoi | Sens |
|---|---|---|
| $di/dt$ | une commutation rapide extrait la charge avant qu'elle ne se recombine | ↗ |
| $V_R$ | une tension inverse forte balaie plus fort et élargit la zone déserte | ↗ |
| $I_F$ | plus de courant direct = plus de porteurs stockés au départ | ↗ |

Le facteur en $V_R$ se recoupe bien avec la littérature : si $Q_{rr}\propto\sqrt{V_R}$
alors $E_{rr} = Q_{rr}V_R \propto V_R^{1,5}$, proche du $V^{1,4}$ couramment rapporté
pour l'énergie de recouvrement.

Conséquence directe : en **ZVS**, $V_R = 0$ annule $Q_{rr}$ — il n'y a rien pour forcer
un recouvrement. Les exposants $a,b,c$ sont des champs de `BODY_DIODE`, à recaler par
composant si la datasheet donne une courbe $Q_{rr}$.

$$\boxed{P_{gate} = Q_g\, \Delta V_{gs}\, f_{sw}}
\qquad\text{réparti}\ \propto R:\quad
P_{g,int} = P_{gate}\frac{R_{g,int}}{R_{G,tot}}$$

**Seule la part interne chauffe la puce.** Le driver et la résistance externe dissipent
la leur hors du boîtier :

$$\boxed{P_{total} = P_{cond} + P_{sw} + P_{oss} + P_{body} + P_{g,int}}$$

### 2.3 Vitesses de commutation

$$\left.\frac{di}{dt}\right|_{on} = \frac{I_{on}}{t_{ri}}
\qquad
\left.\frac{dv}{dt}\right|_{on} = \frac{V_{on}}{t_{fv}}
\qquad
\left.\frac{dv}{dt}\right|_{off} = \frac{V_{off}}{t_{rv}}$$

### 2.4 Couplage thermique

$R_{DS(on)}$ monte avec $T_j$, $T_j$ monte avec les pertes : ni l'un ni l'autre ne se
calcule seul. Point fixe itéré jusqu'à convergence :

$$\boxed{T_j^{(k+1)} = T_{amb} + R_{th}\; P_{total}\!\left(T_j^{(k)}\right)}$$

S'il diverge, le montage est en **emballement thermique** — le notebook l'affiche.

### 2.5 Tensions commutées — la souplesse du modèle

Il n'y a **pas** de bouton « hard/soft switching » : on renseigne directement la tension
réellement balayée sur chaque front.

| Cas | $V_{turn\,on}$ | $V_{turn\,off}$ |
|---|---|---|
| Commutation dure | $V_{bus}$ | $V_{bus}$ |
| ZVS à l'amorçage | $0$ | $V_{bus}$ |
| ZVS des deux côtés | $0$ | $0$ |
| Snubbé au blocage | $V_{bus}$ | fraction de $V_{bus}$ |

Mettre $V_{turn\,on} = 0$ annule tout seul $P_{oss}$ **et** le recouvrement.

---
## 3. Panneau de calcul

Le $Q_{rr}$ propose trois modes :

- **Auto** — la body diode du MOSFET lui-même, mise à l'échelle du $di/dt$ réel.
- **Aucun** — pas de recouvrement (ZCS, ou GaN sans body diode).
- **Diode externe** — on saisit le $Q_{rr}$ de la diode qui se recouvre (Schottky
  parallèle, diode de roue libre discrète...). C'est la valeur que le turn-on doit
  balayer, quelle qu'en soit l'origine.

In [6]:
# --- Panneau de commande -----------------------------------------------------
_ST = {"description_width": "185px"}
_LY = W.Layout(width="380px")


def _f(desc, value, step=None, lo=0.0, hi=1e9):
    """Champ numérique borné. lo/hi évitent les saisies aberrantes."""
    return W.BoundedFloatText(value=value, description=desc, style=_ST, layout=_LY,
                              step=step, min=lo, max=hi)


# Composants
w_mosfet = W.Dropdown(options=sorted(MOSFET_LIBRARY), description="MOSFET :",
                      style=_ST, layout=_LY)
w_driver = W.Dropdown(options=sorted(DRIVER_LIBRARY), description="Driver :",
                      style=_ST, layout=_LY)

# Point de fonctionnement
w_von = _f("V commutée à l'amorçage [V] :", 48.0, 1.0)
w_voff = _f("V commutée au blocage [V] :", 48.0, 1.0)
w_irms = _f("Courant efficace I_rms [A] :", 15.8, 0.5)
w_ion = _f("Courant commuté amorçage [A] :", 25.0, 0.5)
w_ioff = _f("Courant commuté blocage [A] :", 25.0, 0.5)
w_fsw = _f("Fréquence de découpage [kHz] :", 100.0, 10.0)

# Boucle de grille
w_rgon = _f("R grille externe amorçage [Ω] :", 2.2, 0.1)
w_rgoff = _f("R grille externe blocage [Ω] :", 1.0, 0.1)

# Body diode
w_ibody = _f("Courant diode temps mort [A] :", 0.0, 0.5)
w_dbody = _f("Rapport cyclique diode [%] :", 0.0, 0.5, hi=100.0)

# Q_rr
w_qrr_mode = W.Dropdown(
    options=[("Auto — body diode du MOSFET", "auto"),
             ("Aucun — pas de recouvrement", "none"),
             ("Diode externe — saisir Q_rr", "extern")],
    value="auto", description="Recouvrement Q_rr :", style=_ST, layout=_LY,
)
w_qrr_ext = _f("Q_rr diode externe [nC] :", 78.0, 1.0)
w_qrr_ext.disabled = True


def _toggle_qrr(_=None):
    w_qrr_ext.disabled = w_qrr_mode.value != "extern"


w_qrr_mode.observe(_toggle_qrr, names="value")

# Thermique
w_tj = _f("Tj d'évaluation [°C] :", 100.0, 5.0, lo=-40.0, hi=300.0)
w_tamb = _f("Température ambiante [°C] :", 40.0, 5.0, lo=-40.0, hi=200.0)
w_rth = _f("R_th jonction→ambiante [°C/W] :", 20.0, 1.0)
w_rth_auto = W.Checkbox(value=False, description="Utiliser le R_thJA datasheet",
                        indent=False, layout=W.Layout(width="380px"))

w_go = W.Button(description="Calculer", button_style="primary",
                icon="calculator", layout=W.Layout(width="200px", height="38px"))
out = W.Output()


def _section(titre, widgets):
    return W.VBox([W.HTML(f"<b style='font-size:13px'>{titre}</b>"), *widgets],
                  layout=W.Layout(margin="0 28px 14px 0"))


panneau = W.VBox([
    W.HBox([
        _section("Composants", [w_mosfet, w_driver]),
        _section("Point de fonctionnement", [w_von, w_voff, w_fsw]),
    ]),
    W.HBox([
        _section("Courants", [w_irms, w_ion, w_ioff]),
        _section("Boucle de grille", [w_rgon, w_rgoff]),
    ]),
    W.HBox([
        _section("Body diode / recouvrement", [w_ibody, w_dbody, w_qrr_mode, w_qrr_ext]),
        _section("Thermique", [w_tj, w_tamb, w_rth, w_rth_auto]),
    ]),
    w_go,
    out,
])


def _operating_point():
    q_rr = {"auto": None, "none": 0.0, "extern": w_qrr_ext.value * 1e-9}[w_qrr_mode.value]
    return OPERATING_POINT(
        v_turn_on=w_von.value,
        v_turn_off=w_voff.value,
        i_rms=w_irms.value,
        f_sw=w_fsw.value * 1e3,
        i_on=w_ion.value,
        i_off=w_ioff.value,
        r_g_ext_on=w_rgon.value,
        r_g_ext_off=w_rgoff.value,
        i_body=w_ibody.value,
        d_body=w_dbody.value / 100.0,
        q_rr_opposite=q_rr,
    )


def _controles(mosfet, op):
    """Cohérence du point de fonctionnement vis-à-vis des limites du composant."""
    alertes = []
    v_max_coss = mosfet.c_oss.vds_points[-1]
    v_sw = max(op.v_turn_on, op.v_turn_off)

    if v_sw > mosfet.v_dss_max:
        alertes.append(("stop", f"Tension commutée {v_sw:.0f} V au-dessus du V_DSS max "
                                f"du {mosfet.component_info.part_number} "
                                f"({mosfet.v_dss_max:.0f} V) — claquage."))
    if op.v_turn_on > v_max_coss:
        alertes.append(("stop", f"La courbe C_oss de la datasheet s'arrête à "
                                f"{v_max_coss:.0f} V : impossible de calculer P_oss à "
                                f"{op.v_turn_on:.0f} V sans extrapoler. Étendre "
                                f"`vds_points` / `coss_points` dans MOSFET_LIBRARY."))
    for nom, val in (("I_rms", op.i_rms), ("I amorçage", op.i_commutated_on()),
                     ("I blocage", op.i_commutated_off())):
        if val > mosfet.i_max:
            alertes.append(("warn", f"{nom} = {val:.0f} A au-dessus du calibre "
                                    f"({mosfet.i_max:.0f} A)."))
    if w_tj.value > mosfet.thermal.t_j_max:
        alertes.append(("warn", f"Tj d'évaluation {w_tj.value:.0f} °C au-dessus de "
                                f"T_j,max ({mosfet.thermal.t_j_max:.0f} °C)."))
    # I_F de la loi Q_rr = i_body : sans conduction déclarée, pas de charge
    # stockée, donc pas de recouvrement. Cohérent, mais facile à oublier.
    if w_qrr_mode.value == "auto" and op.i_body == 0.0 and op.v_turn_on > 0.0:
        alertes.append(("warn", "Mode Q_rr « Auto » mais courant de diode nul : une "
                                "body diode qui ne conduit jamais n'a rien à recouvrer, "
                                "donc P_rr = 0. Si elle roue libre pendant le temps mort, "
                                "renseigne « Courant diode temps mort » — c'est le I_F "
                                "de la loi Q_rr."))
    return alertes


def calculer(_=None):
    with out:
        clear_output(wait=True)
        mosfet = load_mosfet(w_mosfet.value)
        driver = load_driver(w_driver.value)
        op = _operating_point()

        alertes = _controles(mosfet, op)
        for niveau, message in alertes:
            couleur = "#d03b3b" if niveau == "stop" else "#b26a00"
            prefixe = "Calcul impossible" if niveau == "stop" else "Attention"
            display(HTML(f"<p style='color:{couleur};margin:2px 0'>"
                         f"<b>{prefixe} :</b> {message}</p>"))
        if any(niveau == "stop" for niveau, _ in alertes):
            return

        try:
            res = loss_single_mosfet_at_temp(mosfet, driver, op, t_j=w_tj.value)
        except ValueError as err:
            display(HTML(f"<p style='color:#d03b3b'><b>Calcul impossible :</b> {err}</p>"))
            return

        r_th = None if w_rth_auto.value else w_rth.value
        th = loss_thermal_iteration(mosfet, driver, op, t_ambient=w_tamb.value, r_th=r_th)
        t_j_max = mosfet.thermal.t_j_max

        # -- synthèse
        etat = ("<span style='color:#d03b3b'><b>DIVERGE — emballement thermique</b></span>"
                if not th.converged else
                f"<span style='color:#d03b3b'><b>Tj = {th.t_j:.1f} °C > T_j,max = {t_j_max:.0f} °C</b></span>"
                if th.t_j_max_exceeded else
                f"<span style='color:#0ca30c'><b>Tj = {th.t_j:.1f} °C</b> "
                f"(marge {t_j_max - th.t_j:.0f} °C sous T_j,max)</span>")
        display(HTML(
            f"<div style='font-size:14px;line-height:1.7'>"
            f"<b>{w_mosfet.value}</b> piloté par <b>{w_driver.value}</b><br>"
            f"Pertes à Tj = {w_tj.value:.0f} °C : <b>{res.p_total:.3f} W</b> "
            f"&nbsp;·&nbsp; R_th = {th.r_th:.1f} °C/W &nbsp;·&nbsp; {etat}</div>"
        ))

        # -- tableau (jumeau textuel des graphes : jamais de valeur cachée)
        display(HTML("<b>Bilan de pertes [W]</b>"))
        display(loss_table(res))

        # -- temps et vitesses
        display(HTML(
            "<b>Commutation</b>"
            "<table style='margin-top:6px'>"
            f"<tr><td style='padding:2px 18px 2px 0'>t_ri (montée courant)</td>"
            f"<td><b>{res.t_ri * 1e9:.1f} ns</b></td>"
            f"<td style='padding-left:28px'>di/dt amorçage</td>"
            f"<td><b>{res.di_dt_on / 1e9:.2f} A/ns</b></td></tr>"
            f"<tr><td>t_fv (descente tension)</td><td><b>{res.t_fv * 1e9:.1f} ns</b></td>"
            f"<td style='padding-left:28px'>dv/dt amorçage</td>"
            f"<td><b>{res.dv_dt_on / 1e9:.2f} V/ns</b></td></tr>"
            f"<tr><td>t_rv (montée tension)</td><td><b>{res.t_rv * 1e9:.1f} ns</b></td>"
            f"<td style='padding-left:28px'>di/dt blocage</td>"
            f"<td><b>{res.di_dt_off / 1e9:.2f} A/ns</b></td></tr>"
            f"<tr><td>t_fi (descente courant)</td><td><b>{res.t_fi * 1e9:.1f} ns</b></td>"
            f"<td style='padding-left:28px'>dv/dt blocage</td>"
            f"<td><b>{res.dv_dt_off / 1e9:.2f} V/ns</b></td></tr>"
            f"<tr><td>R_DS(on) à {w_tj.value:.0f} °C</td>"
            f"<td colspan='3'><b>{res.r_ds_on * 1e3:.3f} mΩ</b> "
            f"(soit {res.r_ds_on / mosfet.r_ds_on.r_ds_on_25:.2f} × la valeur à 25 °C)</td></tr>"
            "</table>"
        ))

        # -- pertes hors puce
        display(HTML(
            f"<p style='color:#52514e'>Hors boîtier : {res.p_gate_drv * 1e3:.0f} mW dans le "
            f"driver, {res.p_gate_ext * 1e3:.0f} mW dans R_grille externe "
            f"(non comptés dans P_total).</p>"
        ))

        # -- graphes
        fig = plot_loss_breakdown(
            res,
            title=f"{w_mosfet.value} — bilan de pertes",
            subtitle=f"{w_von.value:.0f} V / {w_ion.value:.0f} A / {w_fsw.value:.0f} kHz "
                     f"à Tj = {w_tj.value:.0f} °C — total {res.p_total:.2f} W",
        )
        display(fig)
        plt.close(fig)

        fig = plot_thermal_iteration(
            th, t_j_max=t_j_max,
            title=f"{w_mosfet.value} — convergence de Tj",
            subtitle=f"R_th = {th.r_th:.0f} °C/W, T_amb = {w_tamb.value:.0f} °C",
        )
        display(fig)
        plt.close(fig)


w_go.on_click(calculer)
display(panneau)
calculer()

---
## 4. Lire le résultat — et ce que le modèle ne dit pas

**Ce qui est vérifié.** L'énergie se conserve : `P_total` est exactement la somme des six
postes, les trois parts de perte grille somment à $Q_g \Delta V_{gs} f_{sw}$, et
$C_{oss,er}$ redonne bien $\int_0^V C_{oss}(v)\,v\,dv$.

**Trois limites à garder en tête :**

1. **$R_{DS(on)}(T_j)$ est linéaire, donc optimiste.** À 175 °C le modèle donne ~1,75 ×
   $R_{25}$ là où une loi en $T^{2,3}$ donne ~2,55 ×, soit **46 % d'écart**. Conséquence
   directe : la boucle thermique converge plus bas que la réalité et l'emballement est
   moins probable dans le modèle qu'en vrai. C'est le paramètre qui fausse tout
   silencieusement — à recaler sur la datasheet via `alpha_R`.

2. **$Q_{rr}$ est extrapolé loin du point de test.** La loi à trois facteurs part des
   conditions datasheet ; à 2 A/ns on est typiquement 20 × au-delà en $di/dt$. Les
   exposants $a = b = c = \tfrac12$ sont les valeurs d'ingénieur usuelles, pas des
   constantes physiques : elles dépendent du composant. Si le recouvrement pèse lourd
   dans le bilan, recale-les sur une courbe datasheet, ou saisis directement la charge
   en mode « diode externe ».

3. **$V_{plateau}$ est pris fixe** alors qu'il dépend du courant ($V_{pl} = V_{th} +
   I_d/g_{fs}$) et de la température. Les temps de commutation en héritent.

**Points de conception à surveiller dans les résultats :**

- Un $dv/dt$ élevé au blocage (> ~5 V/ns) menace l'immunité du composant d'en face
  (réamorçage parasite par $C_{gd}$) et le mode commun à travers l'isolation.
- Si `P_sw` domine, jouer sur $R_{grille}$ ; si c'est `P_cond`, c'est le composant ou le
  refroidissement qu'il faut reprendre.
- La marge sous $T_{j,max}$ doit rester confortable : la loi $R_{DS(on)}$ étant optimiste,
  converger à 5 °C sous la limite n'est **pas** un design validé.